# Falcon-H1-3B-Instruct — Query2Doc Query Generator

**Experiment:** exp_005 — Falcon-H1-3B-Instruct + Dense Retrieval  
**Technique:** Query2Doc (pseudo-document generation)  
**Reference baseline:** exp_003 (Qwen 2.5 3B, NDCG@10=0.5435)

## Two temperature runs:
- `temperature=0.1` → `enhanced_queries_falcon_h1_3b_temp01.pkl` (Falcon recommended)
- `temperature=0.7` → `enhanced_queries_falcon_h1_3b_temp07.pkl` (match Qwen for fair comparison)

## GPU: A100 (40 GB) — Colab Pro+
- **bfloat16** (model-recommended dtype, native on A100)
- **Flash Attention 2** (bypasses SDPA batching bug, faster than eager)
- **batch_size=8** (40 GB VRAM, plenty of headroom)
- **Estimated total time:** ~30-50 minutes (2 × 15-25 min)

---

## Step 1: Install Dependencies

> ⚠️ **Select A100 runtime first:** Runtime → Change runtime type → A100  
> ⚠️ **CRITICAL:** Install transformers from source for `falcon_h1` architecture support  
> ⛔ **DO NOT** install `mamba-ssm` or `causal-conv1d`  
> 🔄 **Restart runtime after this cell!**

In [ ]:
# ── Step 1: Install all dependencies ──────────────────────────────────────────
#
# Runtime: A100 (40 GB) recommended — select in Runtime → Change runtime type
#
# Java is required by pyserini (for MIRACL data loading).
# transformers from SOURCE — required for falcon_h1 architecture (v5.2.0+).
# DO NOT install mamba-ssm or causal-conv1d — not needed and WILL fail on Colab.
#
# After this cell: Runtime → Restart runtime, then continue from Step 2.
# ──────────────────────────────────────────────────────────────────────────────

# 1. Java (required by pyserini)
!apt-get install -qq openjdk-21-jdk-headless

# 2. Retrieval / data loading libraries
!pip install -q pyserini faiss-cpu

# 3. Transformers from source (for falcon_h1 support)
!pip install -q git+https://github.com/huggingface/transformers.git

# 4. ML / utility libraries
!pip install -q torch datasets accelerate tqdm

print("\n" + "="*60)
print("✓ Installation complete")
print("="*60)
print("⚠️  IMPORTANT: Restart runtime now!")
print("   Runtime → Restart runtime")
print("   Then run cells starting from Step 2")
print("="*60)

Selecting previously unselected package openjdk-21-jre-headless:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jre-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
Selecting previously unselected package openjdk-21-jdk-headless:amd64.
Preparing to unpack .../openjdk-21-jdk-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.10+7-1~22.04) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/java to provide /usr/bin/java (java) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/jpackage to provide /usr/bin/jpackage (jpackage) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/keytool to provide /usr/bin/keytool (keytool) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/b

## Step 2: Mount Drive and Setup Environment

> Run this after restarting runtime

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Clone project repo (or pull if already cloned)
!git clone https://github.com/Osmanoor/graduation.git 2>/dev/null || (cd /content/graduation && git pull)
%cd /content/graduation/arabic-rag-query-enhancement

import os
import sys

# Java home required by pyserini
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

import torch
print(f"\n✓ Environment configured")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Verify transformers version supports falcon_h1
import transformers
print(f"Transformers version: {transformers.__version__}")

/content/graduation/arabic-rag-query-enhancement
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)

✓ Environment configured
GPU Available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Transformers version: 5.3.0.dev0


## Step 3: Load MIRACL Arabic Data

In [ ]:
from src.utils.data_loader import MIRACLDataLoader

data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f"\nDataset Statistics:")
print(f"  Queries: {len(query_ids)}")
print(f"  Qrels:   {len(qrels)}")
print(f"\nSample query: {query_texts[0]}")

Loading topics from miracl-v1.0-ar-dev...
✓ Loaded 2896 queries
Loading qrels from miracl-v1.0-ar-dev...
✓ Loaded qrels for 2896 queries

Dataset Statistics:
  Queries: 2896
  Qrels:   2896

Sample query: من هو علي بن محمد السمري؟


## Step 4: Initialize Falcon-H1-3B-Instruct Enhancer (temperature=0.1)

**Model:** `tiiuae/Falcon-H1-3B-Instruct`  
**Architecture:** Hybrid Mamba2-Transformer (native in transformers v5.2.0+)  
**dtype:** bfloat16 (model-recommended, native on A100)  
**Attention:** Flash Attention 2 (bypasses SDPA batching bug)  
**VRAM:** ~10-11 GB → leaves ~29 GB free on A100 (40 GB)  
**Download:** ~6 GB on first run (~2 min)

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_NAME = "tiiuae/Falcon-H1-3B-Instruct"

gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_vram:.0f} GB)")

# Step 1: Create enhancer (loads tokenizer + float16 model)
from src.enhancers.query2doc import Query2DocEnhancer

enhancer = Query2DocEnhancer(
    model_name=MODEL_NAME,
    max_new_tokens=128,
    temperature=0.1,
    top_p=0.9,
    batch_size=1  # Falcon-H1 bug: batched generation crashes on ALL attention backends
)

# Step 2: Replace model with bfloat16 (model-recommended, native on A100)
print("\nSwapping to bfloat16...")
del enhancer.model
torch.cuda.empty_cache()

enhancer.model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
enhancer.model.eval()

free, total = torch.cuda.mem_get_info()
print(f"\nGPU: {(total-free)/1e9:.1f} GB used / {total/1e9:.1f} GB total")
print(f"Free: {free/1e9:.1f} GB")
print(f"\n✓ Falcon-H1-3B-Instruct ready")
print(f"  bfloat16 + single-query mode (Falcon-H1 batching bug)")

GPU: NVIDIA A100-SXM4-40GB (42 GB)
Loading tiiuae/Falcon-H1-3B-Instruct in float16...


Loading weights:   0%|          | 0/547 [00:00<?, ?it/s]

✓ Model loaded on cuda:0
✓ Batch size: 1 (processing 1 queries at once)
✓ Max tokens: 128 (shorter = faster)

Swapping to bfloat16...


Loading weights:   0%|          | 0/547 [00:00<?, ?it/s]


GPU: 13.8 GB used / 42.4 GB total
Free: 28.7 GB

✓ Falcon-H1-3B-Instruct ready
  bfloat16 + single-query mode (Falcon-H1 batching bug)


## Step 5: Sanity Check — First 5 Queries

**Check before proceeding to full run:**
- Output is in Arabic (not English, not garbage/repetition)
- Pseudo-document is relevant to the query topic
- Expansion ratio is reasonable (5–12×)
- No error messages or warnings

In [ ]:
print("Sanity check: testing on first 5 queries (temp=0.1)\n")
print("="*60)

for i in range(5):
    enhanced = enhancer.enhance(query_texts[i], query_ids[i])
    ratio = len(enhanced) / max(len(query_texts[i]), 1)
    print(f"\nQuery {i+1} [{query_ids[i]}]: {query_texts[i]}")
    print(f"Enhanced ({len(enhanced)} chars, {ratio:.1f}x expansion):")
    print(f"{enhanced[:300]}..." if len(enhanced) > 300 else enhanced)

print("\n" + "="*60)
print("✓ Sanity check complete")
print("\nBefore proceeding, verify:")
print("  [ ] Output is in Arabic")
print("  [ ] Content is relevant to query")
print("  [ ] Expansion ratio 5-12x")
print("  [ ] No errors or garbage output")

Sanity check: testing on first 5 queries (temp=0.1)


Query 1 [8099]: من هو علي بن محمد السمري؟
Enhanced (397 chars, 15.9x expansion):
من هو علي بن محمد السمري؟ علي بن محمد السمري (ت 130 هـ / 747 م) كان عالمًا وشاعرًا وفقيهًا إمامًا في الفقه الإسلامي. يُعتبر من أبرز الشخصيات في علم الفقه خلال العصر الأموي. ولد في مدينة السمرة (التي تُعرف اليوم باسم السمايدة) في شبه الجزيرة العربية.

اشتهر علي بن محمد بكتابه "المقدمة" والذي يعد من أ...

Query 2 [3640]: متى تم إستخدام الغوّاصات لأول مرة؟
Enhanced (171 chars, 5.0x expansion):
متى تم إستخدام الغوّاصات لأول مرة؟ تم استخدام الغواصات لأول مرة في القرن التاسع عشر، حيث بدأت السفن البحرية بتطوير وتصميم السفن الغواصة لتجنب اكتشافها من قبل السفن الأخرى.

Query 3 [4971]: من هو القديس المسمى بالصخرة؟
Enhanced (298 chars, 10.6x expansion):
من هو القديس المسمى بالصخرة؟ القديس المسمى بالصخرة هو القديس يوحنا المعمدان، المعروف أيضًا باسم يوحنا المعمد. يُعتبر ابن زكريا (عبر الإنجيل) ويُعتبر المعمد الأول للمسيح وفقًا للثقافة الكاثوليكية والبروتستانتية. يُذ

## Step 6: Full Batch Generation — temperature=0.1

**Expected time:** ~15-25 minutes on A100 with batch_size=8  
Flash Attention 2 bypasses the SDPA batching bug that crashed on T4.  

**Fallback:** If this still fails, replace `enhancer.enhance_batch(...)` with the
single-query loop from the T4 fallback (see Step 4b cell).

In [ ]:
import time
from tqdm.notebook import tqdm

print("="*60)
print("FULL RUN: temperature=0.1 (Falcon recommended setting)")
print(f"Queries: {len(query_texts)}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Mode: bfloat16, single-query (Falcon-H1 batching bug)")
print(f"Estimated time: ~30-40 minutes on A100")
print("="*60 + "\n")

start_01 = time.time()
enhanced_queries_01 = []

for qtext, qid in tqdm(zip(query_texts, query_ids),
                        total=len(query_texts),
                        desc="Enhancing queries (temp=0.1)"):
    enhanced = enhancer.enhance(qtext, qid)
    enhanced_queries_01.append(enhanced)

elapsed_01 = time.time() - start_01
print(f"\n✓ Enhanced {len(enhanced_queries_01)} queries in {elapsed_01/60:.1f} minutes (temp=0.1)")
print(f"  Speed: {len(enhanced_queries_01) / (elapsed_01/60):.1f} queries/minute")

FULL RUN: temperature=0.1 (Falcon recommended setting)
Queries: 2896
GPU: NVIDIA A100-SXM4-40GB
Mode: bfloat16, single-query (Falcon-H1 batching bug)
Estimated time: ~30-40 minutes on A100



Enhancing queries (temp=0.1):   0%|          | 0/2896 [00:00<?, ?it/s]


✓ Enhanced 2896 queries in 282.6 minutes (temp=0.1)
  Speed: 10.2 queries/minute


## Step 7: Save temperature=0.1 Results

In [ ]:
import pickle
from datetime import datetime

data_01 = {
    'query_ids': query_ids,
    'original': query_texts,
    'enhanced': enhanced_queries_01,
    'metadata': {
        'model': 'tiiuae/Falcon-H1-3B-Instruct',
        'architecture': 'Hybrid Mamba2-Transformer (falcon_h1)',
        'temperature': 0.1,
        'max_new_tokens': 128,
        'batch_size': 8,
        'dtype': 'bfloat16',
        'attn_implementation': 'eager',
        'gpu': torch.cuda.get_device_name(0),
        'technique': 'query2doc',
        'dataset': 'miracl-ar-dev',
        'date': datetime.now().isoformat(),
        'num_queries': len(query_ids),
        'runtime_minutes': round(elapsed_01 / 60, 1)
    }
}

# Save locally in Colab
local_path_01 = 'enhanced_queries_falcon_h1_3b_temp01.pkl'
with open(local_path_01, 'wb') as f:
    pickle.dump(data_01, f)
print(f"✓ Saved locally: {local_path_01}")

# Save to Google Drive for persistence
drive_base = '/content/drive/MyDrive/graduation project/colab_data'
drive_path_01 = f'{drive_base}/enhanced_queries_falcon_h1_3b_temp01.pkl'
with open(drive_path_01, 'wb') as f:
    pickle.dump(data_01, f)
print(f"✓ Saved to Drive: {drive_path_01}")
print(f"\nMetadata: {data_01['metadata']}")

✓ Saved locally: enhanced_queries_falcon_h1_3b_temp01.pkl


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/graduation project/colab_data/enhanced_queries_falcon_h1_3b_temp01.pkl'

## Step 8: Re-run with temperature=0.7

**Purpose:** Fair cross-model comparison with Qwen 2.5 3B (exp_003 used temperature=0.7)  
**Note:** No need to reload the model — just update the temperature attribute  
**Expected time:** ~40 minutes

In [ ]:
# Change temperature — no model reload needed
enhancer.temperature = 0.7
print(f"Temperature updated to: {enhancer.temperature}")

print("\n" + "="*60)
print("FULL RUN: temperature=0.7 (match Qwen baseline for fair comparison)")
print(f"Queries: {len(query_texts)}")
print(f"Batch size: {enhancer.batch_size}")
print(f"Estimated time: ~15-25 minutes on A100")
print("="*60 + "\n")

start_07 = time.time()

enhanced_queries_07 = enhancer.enhance_batch(
    query_texts,
    query_ids,
    show_progress=True
)

elapsed_07 = time.time() - start_07
print(f"\n✓ Enhanced {len(enhanced_queries_07)} queries in {elapsed_07/60:.1f} minutes (temp=0.7)")
print(f"  Speed: {len(enhanced_queries_07) / (elapsed_07/60):.1f} queries/minute")

Temperature updated to: 0.7

FULL RUN: temperature=0.7 (match Qwen baseline for fair comparison)
Queries: 2896
Batch size: 1
Estimated time: ~15-25 minutes on A100




Enhancing batches:   0%|          | 2/2896 [00:29<11:43:31, 14.59s/it]


KeyboardInterrupt: 

## Step 9: Save temperature=0.7 Results

In [ ]:
data_07 = {
    'query_ids': query_ids,
    'original': query_texts,
    'enhanced': enhanced_queries_07,
    'metadata': {
        'model': 'tiiuae/Falcon-H1-3B-Instruct',
        'architecture': 'Hybrid Mamba2-Transformer (falcon_h1)',
        'temperature': 0.7,
        'max_new_tokens': 128,
        'batch_size': 8,
        'dtype': 'bfloat16',
        'attn_implementation': 'eager',
        'gpu': torch.cuda.get_device_name(0),
        'technique': 'query2doc',
        'dataset': 'miracl-ar-dev',
        'date': datetime.now().isoformat(),
        'num_queries': len(query_ids),
        'runtime_minutes': round(elapsed_07 / 60, 1)
    }
}

local_path_07 = 'enhanced_queries_falcon_h1_3b_temp07.pkl'
with open(local_path_07, 'wb') as f:
    pickle.dump(data_07, f)
print(f"✓ Saved locally: {local_path_07}")

drive_path_07 = f'{drive_base}/enhanced_queries_falcon_h1_3b_temp07.pkl'
with open(drive_path_07, 'wb') as f:
    pickle.dump(data_07, f)
print(f"✓ Saved to Drive: {drive_path_07}")
print(f"\nMetadata: {data_07['metadata']}")

## Step 10: Expansion Statistics Comparison

Quick summary of both runs before handing off to the Evaluator notebook.

In [ ]:
import numpy as np

print("=" * 60)
print("EXPANSION STATISTICS SUMMARY")
print("=" * 60)

orig_lens = [len(q) for q in query_texts]

for label, queries in [("temp=0.1 (Falcon optimal)", enhanced_queries_01),
                        ("temp=0.7 (Qwen comparison)", enhanced_queries_07)]:
    enh_lens = [len(q) for q in queries]
    ratios = [e / max(o, 1) for e, o in zip(enh_lens, orig_lens)]
    print(f"\n{label}:")
    print(f"  Avg original length : {np.mean(orig_lens):.1f} chars")
    print(f"  Avg enhanced length : {np.mean(enh_lens):.1f} chars")
    print(f"  Avg expansion ratio : {np.mean(ratios):.2f}x")
    print(f"  Median expansion    : {np.median(ratios):.2f}x")

print("\n" + "=" * 60)
print("REFERENCE (exp_003, Qwen 2.5 3B, temp=0.7):")
print("  Avg expansion ratio: 9.73x  (Avg enhanced: 247.6 chars)")
print("=" * 60)

print("\n✓ Both pkl files saved. Next step:")
print("  Upload to evaluate_enhanced_queries.ipynb for Dense retrieval evaluation")
print("  Files:")
print(f"    - {local_path_01}")
print(f"    - {local_path_07}")

---

## Next Steps

1. Open `experiments/evaluate_enhanced_queries.ipynb`
2. Upload `enhanced_queries_falcon_h1_3b_temp01.pkl` → run Dense evaluation → record metrics
3. Upload `enhanced_queries_falcon_h1_3b_temp07.pkl` → run Dense evaluation → record metrics
4. Document results in `docs/experiments/exp_005_falcon_h1_3b_dense.md`

**Reference metrics to beat (exp_003, Qwen 2.5 3B):**

| Metric | Baseline | Qwen 2.5 3B |
|--------|----------|-------------|
| NDCG@10 | 0.4993 | 0.5435 |
| Recall@10 | 0.6156 | 0.6608 |
| Recall@100 | 0.8407 | 0.8594 |
| MRR | 0.5328 | 0.5742 |